In [ ]:
# Set up the environment - note see MOT_transformer_model_module_test for the full list of imports
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
import os
import gc
import warnings

script_path = os.getcwd()

# Ignore warnings
warnings.filterwarnings('ignore')

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

# Enable auto updates from imported modules
%load_ext autoreload
%autoreload 2


In [ ]:
# 1. Read in training data

print("Reading in training data...")
data_fuels = []
for veh_name in ["car_diesel", "car_petrol"]:
    print(f"\tReading in {veh_name} data...")
    fname = f"transformer_training_data_{veh_name}.parquet"
    fpath = os.path.join(script_path, "data", "transformer_data", fname)
    data = pd.read_parquet(fpath, engine='pyarrow')
    
    # Only keep the columns we need for training
    categorical_cols = ['fuel_type', 'last_test', ]
    numerical_cols = ['mileage_per_year', 'test_mileage', 'age_year', 'time_between_tests']
    training_cols = ['vehicle_id'] + categorical_cols + numerical_cols
    data = data[training_cols]
    
    print(f"\tSample: {data['vehicle_id'].nunique()}")

    data_fuels.append(data)
    
data = pd.concat(data_fuels, ignore_index=True)

print("Combined Sample: ", data['vehicle_id'].nunique())

del data_fuels
gc.collect()

print(f"Data loaded. Shape: {data.shape}")
print(data.describe().to_string())
print(data.info())

In [ ]:
# 2. Prepare training data 

from MOT_transformer_model_module_test import prepare_training_data, validate_dataset

batch_size= 15_000 

# Prepare data 
train_dataset, test_dataset, label_encoders, scaler = prepare_training_data(
    data, 
    test_size=0.2, 
    batch_size=batch_size
)

# Create data loaders
train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        pin_memory=True
    )
test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        pin_memory=True
    )

print("Validating training data:")
validate_dataset(train_loader)
print("\nValidating test data:")
validate_dataset(test_loader)
print()

# Save label encoders and scaler
torch.save(label_encoders, "label_encoders.pt")
torch.save(scaler, "scaler.pt")

# Save the training and test data loaders
torch.save(train_loader, "train_loader.pt")
torch.save(test_loader, "test_loader.pt")


In [ ]:
# 3. Set up model 

from MOT_transformer_model_module_test import VehicleTransformer

# Set up model

# Model parameters
input_dim = len(data.columns) - 1  # Subtract 1 for vehicle_id column that is dropped during sequence creation
d_model = 128
nhead = 8
num_layers = 6
dim_feedforward = 256
num_epochs = 4

# Initialize model
model = VehicleTransformer(
    input_dim=input_dim,
    d_model=d_model,
    nhead=nhead,
    num_layers=num_layers,
    dim_feedforward=dim_feedforward
).to(device)


In [ ]:
# 4. Train model

from MOT_transformer_model_module_test import train_model

# load the loaders
train_loader = torch.load("train_loader.pt")
test_loader = torch.load("test_loader.pt")

train_metrics, val_metrics, best_threshold, threshold_results = train_model(
    model, train_loader, test_loader, num_epochs=num_epochs, device=device
)


In [ ]:
# 5. Analyze model predictions on test data

from MOT_transformer_model_module_test import analyze_model_predictions, transformer_figure

# Load the loader, label encoders and scaler - don't need to rerun previous cells
test_loader = torch.load("test_loader.pt")
label_encoders = torch.load("label_encoders.pt")
scaler = torch.load("scaler.pt")

results = analyze_model_predictions(model, test_loader, scaler, device, best_threshold=0.3)
fig = transformer_figure(results)


In [ ]:
# 6. Read in data for predictions:

print("Reading in prediction data...")
data_fuels = []
for veh_name in ["car_diesel", "car_petrol", "car_bev", "car_hev"]:
    print(f"\tReading in {veh_name} data...")
    folder = os.path.join(script_path, "data", "transformer_data")
    data = pd.read_parquet(os.path.join(folder, f"transformer_prediction_data_{veh_name}.parquet"), engine='pyarrow')
    
    # Only keep the columns we need for predictions
    categorical_cols = ['fuel_type', 'last_test']
    numerical_cols = ['mileage_per_year', 'test_mileage', 'age_year', 'time_between_tests']#, 'test_mileage_age_indicator', 'mileage_per_year_age_indicator', 'taxi_indicator']
    predictions_cols = ['vehicle_id', 'test_year', 'make', 'model',
                     'first_use_year', 'fuel efficiency Wh/mi', 'battery capacity (kWh)',
                     'CO2 g/km', 'mass (kg)'
                     ] + categorical_cols + numerical_cols
    data = data[predictions_cols]
    
    data_fuels.append(data)
    
data = pd.concat(data_fuels, ignore_index=True)

print("Combined Sample: ", data['vehicle_id'].nunique())

# Next show the distribution of fuel types
for fuel in ["DI", "PE", "EL", "HY"]:
    n = data.loc[data['fuel_type'] == fuel, 'vehicle_id'].nunique()
    print(f"{fuel}: {n} vehicles")
    
# Change "EL" and "HY" to "DI" or "PE" based upon mileage
# First create a copy of the original fuel type column
data["original_fuel_type"] = data["fuel_type"]
# Catgorise EL and HY based upon mileage criteria:
for first_use_year in range(2005, 2023, 1):
    di_mileage = data.loc[(data["fuel_type"] == "DI") & (data["first_use_year"] == first_use_year) & (data["last_test"] == True), "test_mileage"].mean()
    pe_mileage = data.loc[(data["fuel_type"] == "PE") & (data["first_use_year"] == first_use_year) & (data["last_test"] == True), "test_mileage"].mean()
    cut_off = (di_mileage + pe_mileage) / 2
    print(f"Cut-off for {first_use_year}: {round(cut_off)} miles")

    # Get the ids of the HY and EL that are now DI and PE for this first_use_year
    el_di_ids = data.loc[(data["fuel_type"] == "EL") & (data["first_use_year"] == first_use_year) & (data["last_test"] == True) & (data["test_mileage"] >= cut_off), "vehicle_id"].unique()
    el_pe_ids = data.loc[(data["fuel_type"] == "EL") & (data["first_use_year"] == first_use_year) & (data["last_test"] == True) & (data["test_mileage"] < cut_off), "vehicle_id"].unique()
    hy_di_ids = data.loc[(data["fuel_type"] == "HY") & (data["first_use_year"] == first_use_year) & (data["last_test"] == True) & (data["test_mileage"] >= cut_off), "vehicle_id"].unique()
    hy_pe_ids = data.loc[(data["fuel_type"] == "HY") & (data["first_use_year"] == first_use_year) & (data["last_test"] == True) & (data["test_mileage"] < cut_off), "vehicle_id"].unique()
    
    # Now set these ids as DI or PE
    data.loc[data["vehicle_id"].isin(el_di_ids), "fuel_type"] = "DI"
    data.loc[data["vehicle_id"].isin(el_pe_ids), "fuel_type"] = "PE"
    data.loc[data["vehicle_id"].isin(hy_di_ids), "fuel_type"] = "DI"
    data.loc[data["vehicle_id"].isin(hy_pe_ids), "fuel_type"] = "PE"
    
# Finally check if there are any vehicles that have not been reclassified
n_el = data.loc[data["fuel_type"] == "EL", "vehicle_id"].nunique()
n_hy = data.loc[data["fuel_type"] == "HY", "vehicle_id"].nunique()
print(f"EL: {n_el} vehicles not reclassified")
print(f"HY: {n_hy} vehicles not reclassified")
# defualt these to DI
data.loc[data["fuel_type"] == "EL", "fuel_type"] = "DI"
data.loc[data["fuel_type"] == "HY", "fuel_type"] = "DI"

# Now print the number of ELs now classified as DI and PE
el_de = data.loc[(data["original_fuel_type"] == "EL") &
                (data["fuel_type"] == "DI"), "vehicle_id"].nunique()
el_pe = data.loc[(data["original_fuel_type"] == "EL") &
                (data["fuel_type"] == "PE"), "vehicle_id"].nunique()
hy_de = data.loc[(data["original_fuel_type"] == "HY") &
                (data["fuel_type"] == "DI"), "vehicle_id"].nunique()
hy_pe = data.loc[(data["original_fuel_type"] == "HY") &
                (data["fuel_type"] == "PE"), "vehicle_id"].nunique()
de_original = data.loc[data["original_fuel_type"] == "DI", "vehicle_id"].nunique()
pe_original = data.loc[data["original_fuel_type"] == "PE", "vehicle_id"].nunique()
print(f"EL: {el_de} DI, {el_pe} PE")
print(f"HY: {hy_de} DI, {hy_pe} PE")
print(f"DI: {de_original} original, {el_de + hy_de} reclassified")
print(f"PE: {pe_original} original, {el_pe + hy_pe} reclassified")

# Finally reset last_test for all tests after 2022 to be False
data.loc[data['test_year'] >= 2022, 'last_test'] = False

del data_fuels
gc.collect()

print(f"Data loaded. Shape: {data.shape}")
print(data.describe().to_string())
print(data.info())

In [ ]:
# 7. Predctions
from MOT_transformer_model_module_test import generate_predictions
from MOT_transformer_model_module_test import VehicleTransformer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if device.type == 'cuda':
    torch.cuda.empty_cache()

model = VehicleTransformer(
    input_dim=6,
    d_model=128,
    nhead=8,
    num_layers=6,
    dim_feedforward=256,
).to(device)

model.load_state_dict(torch.load('best_model.pth', map_location=device))
model.eval()

scaler = torch.load('scaler.pt')
label_encoders = torch.load('label_encoders.pt')

data['simulated_data'] = False
data['scrap_probability'] = 0.0

# First remove vehicles that have already been scrapped
ids = data.loc[data["last_test"]==True, "vehicle_id"].unique()
scrapped_data = data[data["vehicle_id"].isin(ids)] 
data = data[~data["vehicle_id"].isin(ids)]
# output the scrapped data
fname = "simulation_data_0_revision.csv"
fpath = os.path.join("data", "simulation_data", fname)
scrapped_data.to_csv(fpath, index=False)

# Cumulative exited vehicles dataframe (built up across iterations)
all_exited_df = scrapped_data.copy()

def print_fleet_summary(data, all_exited_df):
    """Print remaining fleet by original_fuel_type x age, and exited vehicle summaries."""
    fuels = ["DI", "PE", "EL", "HY"]

    # --- Remaining fleet: vehicle counts by age x fuel type ---
    # One row per vehicle (use last record per vehicle to avoid double counting)
    latest = (
        data
        .sort_values(['vehicle_id', 'age_year'])
        .groupby('vehicle_id', as_index=False)
        .last()
    )
    latest['age_year_int'] = latest['age_year'].round().astype(int)

    count_pivot = (
        latest
        .groupby(['age_year_int', 'original_fuel_type'])['vehicle_id']
        .count()
        .unstack(fill_value=0)
        .reindex(columns=fuels, fill_value=0)
    )
    count_pivot['Total'] = count_pivot.sum(axis=1)

    # --- Remaining fleet: cumulative mileage by age x fuel type ---
    mileage_pivot = (
        latest
        .groupby(['age_year_int', 'original_fuel_type'])['test_mileage']
        .mean()
        .round(0)
        .unstack(fill_value=0)
        .reindex(columns=fuels, fill_value=0)
    )

    print("\nRemaining fleet — vehicle count by age and fuel type:")
    print(count_pivot.to_string())

    print("\nRemaining fleet — mean cumulative mileage (miles) by age and fuel type:")
    print(mileage_pivot.to_string())

    # --- Exited vehicles summaries ---
    if len(all_exited_df) > 0:
        exited_last = (
            all_exited_df
            .sort_values(['vehicle_id', 'age_year'])
            .groupby('vehicle_id', as_index=False)
            .last()
        )
        exited_last['age_year_int'] = exited_last['age_year'].round().astype(int)

        exited_age_pivot = (
            exited_last
            .groupby(['age_year_int', 'original_fuel_type'])['test_mileage']
            .mean()
            .round(0)
            .unstack(fill_value=0)
            .reindex(columns=fuels, fill_value=0)
        )
        print("\nExited vehicles — mean cumulative mileage at exit by age and fuel type:")
        print(exited_age_pivot.to_string())

        exited_count_pivot = (
            exited_last
            .groupby(['age_year_int', 'original_fuel_type'])['vehicle_id']
            .count()
            .unstack(fill_value=0)
            .reindex(columns=fuels, fill_value=0)
        )
        exited_count_pivot['Total'] = exited_count_pivot.sum(axis=1)
        print("\nExited vehicles — count at exit by age and fuel type:")
        print(exited_count_pivot.to_string())

        exited_overall = (
            exited_last
            .groupby('original_fuel_type')['test_mileage']
            .mean()
            .round(0)
            .reindex(fuels)
        )
        print("\nExited vehicles — mean cumulative mileage across all ages by fuel type:")
        print(exited_overall.to_string())

        exited_mean_age_overall = (
            exited_last
            .groupby('original_fuel_type')['age_year']
            .mean()
            .round(2)
            .reindex(fuels)
        )
        print("\nExited vehicles — mean age at exit across all ages by fuel type:")
        print(exited_mean_age_overall.to_string())


# Simulate the next test for all vehicles  
i = 1
while len(data) > 1:
    print(f"\nIteration {i}")
    print(f"Data shape: {data.shape}")
    print(f"Unique vehicles: {data['vehicle_id'].nunique()}")
    
    # Generate predictions for next tests
    # batch size much larger than training as we are not backpropagating
    # Note threshold of 0.22 was determined to provide the best performance on the test set
    new_data = generate_predictions(model, data, label_encoders, scaler, device, batch_size=50_000, threshold=0.22, num_workers=0)
    
    # Concatenate with existing data and sort
    data = pd.concat([data, new_data], ignore_index=True)
    data = data.sort_values(['vehicle_id', 'test_year']).reset_index(drop=True)
    
    # Filter out vehicles that have been predicted to be scrapped and export them
    scrapped_ids = data.loc[data["last_test"]==True, "vehicle_id"].unique()
    data_scrapped = data[data["vehicle_id"].isin(scrapped_ids)]
    fname = f"simulation_data_{i}_revision.csv"
    fpath = os.path.join("data", "simulation_data", fname)
    data_scrapped.to_csv(fpath, index=False)

    # Add newly exited vehicles to the cumulative exited dataframe
    all_exited_df = pd.concat([all_exited_df, data_scrapped], ignore_index=True)
    
    # Briefly summarise the scrapped vehicle data
    print(f"Scrapped vehicles: {len(scrapped_ids)}")
    for fuel in ["DI", "PE", "EL", "HY"]:
        n = data_scrapped.loc[data_scrapped["original_fuel_type"]==fuel, "vehicle_id"].nunique()
        print(f"{fuel}: {n}")
    
    # Show the scrapped vehicles values counts by age
    print("Scrapped vehicles by age:")
    print(data_scrapped.loc[data["last_test"]==True, 'age_year'].round().value_counts().sort_index())
    # And finally the mileage distribution by fuel type
    print("Mileage by fuel type:")
    for fuel in ["DI", "PE", "EL", "HY"]:
        print(f"{fuel}: {data_scrapped.loc[(data_scrapped['original_fuel_type'] == fuel) & (data['last_test']==True), 'test_mileage'].mean()}")

    # Keep remaining vehicles for next round of predictions
    data = data[~data["vehicle_id"].isin(scrapped_ids)]

    # Fleet summary for remaining + cumulative exited vehicles
    print_fleet_summary(data, all_exited_df)
    
    # Break if iteration limit reached
    if i == 30:
        fname = "simulation_data_end_revision.csv"
        fpath = os.path.join("data", "simulation_data", fname)
        data.to_csv(fpath, index=False)
        break
    
    i += 1


In [ ]:
# Aggregate all the simulation data
i = 22
print("Aggregating simulation data...")
df = []
for i in range(0, 22, 1):
    print(f"\tReading in simulation_data_{i}.csv")
    fname = f"simulation_data_{i}_revision.csv"
    fpath = os.path.join("data", "simulation_data", fname)
    df.append(pd.read_csv(fpath))

print("Outputting aggregated scrapped data...")
df = pd.concat(df, ignore_index=True)
fname = "simulation_data_2025.csv"
fpath = os.path.join("data", "simulation_data", fname)
df.to_csv(fpath, index=False)

In [ ]:
# 8. Prepare BEV/HEV holdout validation data

import pandas as pd

print("Preparing BEV/HEV holdout validation data...")

validation_fuels = ["car_bev", "car_hev"]
validation_frames = []

for veh_name in validation_fuels:
    print(f"\tReading in {veh_name} data...")
    folder = os.path.join(script_path, "data", "transformer_data")
    df_fuel = pd.read_parquet(os.path.join(folder, f"transformer_prediction_data_{veh_name}.parquet"), engine="pyarrow")

    categorical_cols = ['fuel_type', 'last_test']
    numerical_cols = ['mileage_per_year', 'test_mileage', 'age_year', 'time_between_tests']
    validation_cols = [
        'vehicle_id', 'test_year', 'make', 'model',
        'first_use_year', 'fuel efficiency Wh/mi', 'battery capacity (kWh)',
        'CO2 g/km', 'mass (kg)'
    ] + categorical_cols + numerical_cols

    validation_frames.append(df_fuel[validation_cols].copy())

validation_data = pd.concat(validation_frames, ignore_index=True)
validation_data['original_fuel_type'] = validation_data['fuel_type']
validation_data['raw_last_test'] = validation_data['last_test'].copy()

print(f"Combined validation sample: {validation_data['vehicle_id'].nunique():,} vehicles")
print("Original fuel type distribution:")
for fuel in ["EL", "HY"]:
    n = validation_data.loc[validation_data['original_fuel_type'] == fuel, 'vehicle_id'].nunique()
    print(f"\t{fuel}: {n:,} vehicles")

# Read diesel/petrol reference data to reproduce the same EL/HY -> DI/PE reclassification logic
reference_frames = []
for veh_name in ["car_diesel", "car_petrol"]:
    df_ref = pd.read_parquet(os.path.join(folder, f"transformer_prediction_data_{veh_name}.parquet"), engine="pyarrow")
    reference_cols = ['vehicle_id', 'test_year', 'first_use_year', 'fuel_type', 'last_test', 'test_mileage']
    reference_frames.append(df_ref[reference_cols].copy())

reclassification_reference = pd.concat(reference_frames, ignore_index=True)
reclassification_reference = reclassification_reference.sort_values(['vehicle_id', 'test_year']).reset_index(drop=True)

# Reclassify EL/HY into DI/PE so the trained label encoders can be reused safely
for first_use_year in range(2005, 2021):
    di_mileage = reclassification_reference.loc[
        (reclassification_reference['fuel_type'] == 'DI') &
        (reclassification_reference['first_use_year'] == first_use_year) &
        (reclassification_reference['last_test'] == True),
        'test_mileage'
    ].mean()
    pe_mileage = reclassification_reference.loc[
        (reclassification_reference['fuel_type'] == 'PE') &
        (reclassification_reference['first_use_year'] == first_use_year) &
        (reclassification_reference['last_test'] == True),
        'test_mileage'
    ].mean()

    cut_off = (di_mileage + pe_mileage) / 2

    if pd.isna(cut_off):
        continue

    el_di_ids = validation_data.loc[
        (validation_data['fuel_type'] == 'EL') &
        (validation_data['first_use_year'] == first_use_year) &
        (validation_data['raw_last_test'] == True) &
        (validation_data['test_mileage'] >= cut_off),
        'vehicle_id'
    ].unique()
    el_pe_ids = validation_data.loc[
        (validation_data['fuel_type'] == 'EL') &
        (validation_data['first_use_year'] == first_use_year) &
        (validation_data['raw_last_test'] == True) &
        (validation_data['test_mileage'] < cut_off),
        'vehicle_id'
    ].unique()
    hy_di_ids = validation_data.loc[
        (validation_data['fuel_type'] == 'HY') &
        (validation_data['first_use_year'] == first_use_year) &
        (validation_data['raw_last_test'] == True) &
        (validation_data['test_mileage'] >= cut_off),
        'vehicle_id'
    ].unique()
    hy_pe_ids = validation_data.loc[
        (validation_data['fuel_type'] == 'HY') &
        (validation_data['first_use_year'] == first_use_year) &
        (validation_data['raw_last_test'] == True) &
        (validation_data['test_mileage'] < cut_off),
        'vehicle_id'
    ].unique()

    validation_data.loc[validation_data['vehicle_id'].isin(el_di_ids), 'fuel_type'] = 'DI'
    validation_data.loc[validation_data['vehicle_id'].isin(el_pe_ids), 'fuel_type'] = 'PE'
    validation_data.loc[validation_data['vehicle_id'].isin(hy_di_ids), 'fuel_type'] = 'DI'
    validation_data.loc[validation_data['vehicle_id'].isin(hy_pe_ids), 'fuel_type'] = 'PE'

# Default any remaining unreclassified vehicles to DI, matching the simulation pipeline
validation_data.loc[validation_data['fuel_type'] == 'EL', 'fuel_type'] = 'DI'
validation_data.loc[validation_data['fuel_type'] == 'HY', 'fuel_type'] = 'DI'

# Apply the same censoring rule as cell 6:
# tests in 2022+ are not treated as genuinely scrapped observations
validation_data.loc[validation_data['test_year'] >= 2022, 'last_test'] = False

required_cols = [
    'vehicle_id', 'test_year', 'fuel_type', 'last_test',
    'mileage_per_year', 'test_mileage', 'age_year', 'time_between_tests'
]

n_before_nan = validation_data['vehicle_id'].nunique()
ids_with_nan = validation_data.loc[validation_data[required_cols].isnull().any(axis=1), 'vehicle_id'].unique()
validation_data = validation_data[~validation_data['vehicle_id'].isin(ids_with_nan)].copy()
n_after_nan = validation_data['vehicle_id'].nunique()

validation_data = validation_data.sort_values(['vehicle_id', 'test_year', 'age_year', 'test_mileage']).reset_index(drop=True)
validation_data['test_index'] = validation_data.groupby('vehicle_id').cumcount() + 1
validation_data['sequence_length'] = validation_data.groupby('vehicle_id')['vehicle_id'].transform('size')

eligible_ids = validation_data.loc[validation_data['sequence_length'] >= 2, 'vehicle_id'].unique()
validation_data = validation_data[validation_data['vehicle_id'].isin(eligible_ids)].copy()
n_after_length = validation_data['vehicle_id'].nunique()

validation_targets_df = (
    validation_data
    .groupby('vehicle_id', group_keys=False)
    .tail(1)
    .copy()
    .rename(columns={
        'test_year': 'actual_test_year',
        'mileage_per_year': 'actual_mileage_per_year',
        'test_mileage': 'actual_test_mileage',
        'age_year': 'actual_age_year',
        'time_between_tests': 'actual_time_between_tests',
        'last_test': 'actual_last_test',
        'raw_last_test': 'actual_last_test_raw',
        'fuel_type': 'model_input_fuel_type'
    })
)

validation_targets_df['held_out_sequence_length'] = validation_targets_df['sequence_length']
validation_targets_df = validation_targets_df[[
    'vehicle_id', 'original_fuel_type', 'model_input_fuel_type', 'make', 'model',
    'first_use_year', 'fuel efficiency Wh/mi', 'battery capacity (kWh)',
    'CO2 g/km', 'mass (kg)', 'actual_test_year', 'actual_mileage_per_year',
    'actual_test_mileage', 'actual_age_year', 'actual_time_between_tests',
    'actual_last_test', 'actual_last_test_raw', 'held_out_sequence_length'
]].reset_index(drop=True)

validation_observed_df = validation_data.loc[
    validation_data['test_index'] < validation_data['sequence_length']
].copy().reset_index(drop=True)
validation_observed_df['simulated_data'] = False
validation_observed_df['scrap_probability'] = 0.0

observed_counts = validation_observed_df.groupby('vehicle_id').size()
validation_targets_df['observed_sequence_length'] = validation_targets_df['vehicle_id'].map(observed_counts).fillna(0).astype(int)

print(f"Dropped {n_before_nan - n_after_nan:,} vehicles because of NaN values in required columns")
print(f"Dropped {n_after_nan - n_after_length:,} vehicles because they had fewer than 2 tests")
print(f"Validation vehicles retained: {validation_targets_df['vehicle_id'].nunique():,}")
print(f"Observed history rows: {len(validation_observed_df):,}")
print(f"Held-out target rows: {len(validation_targets_df):,}")

print("\nHeld-out original fuel type distribution:")
print(validation_targets_df['original_fuel_type'].value_counts(dropna=False).to_string())

print("\nHeld-out actual last_test distribution after censoring:")
print(validation_targets_df['actual_last_test'].value_counts(dropna=False).to_string())

print("\nHeld-out raw last_test distribution before censoring:")
print(validation_targets_df['actual_last_test_raw'].value_counts(dropna=False).to_string())

print("\nObserved sequence length summary:")
print(validation_targets_df['observed_sequence_length'].describe().to_string())

print("\nActual test year summary:")
print(validation_targets_df['actual_test_year'].describe().to_string())

validation_targets_df.head()


In [ ]:
# 9. Predict held-out BEV/HEV final tests and compare with actual outcomes

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    median_absolute_error,
    r2_score,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

from MOT_transformer_model_module_test import VehicleTransformer, generate_predictions

validation_threshold = 0.50

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Generating holdout predictions...")
print(f"Using device: {device}")

if device.type == 'cuda':
    torch.cuda.empty_cache()

# Recreate the trained architecture so the saved state dict can be loaded
model = VehicleTransformer(
    input_dim=6,
    d_model=128,
    nhead=8,
    num_layers=6,
    dim_feedforward=256,
).to(device)

model.load_state_dict(torch.load('best_model.pth', map_location=device))
model = model.to(device)
model.eval()

scaler = torch.load('scaler.pt')
label_encoders = torch.load('label_encoders.pt')

validation_predictions_df = generate_predictions(
    model,
    validation_observed_df.copy(),
    label_encoders,
    scaler,
    device,
    batch_size=50_000,
    threshold=validation_threshold,
    num_workers=0,
)

validation_results_df = validation_predictions_df.merge(
    validation_targets_df,
    on='vehicle_id',
    how='inner',
    validate='one_to_one',
    suffixes=('_pred', '_actual')
)

if 'original_fuel_type' not in validation_results_df.columns:
    if 'original_fuel_type_actual' in validation_results_df.columns:
        validation_results_df['original_fuel_type'] = validation_results_df['original_fuel_type_actual']
    elif 'original_fuel_type_pred' in validation_results_df.columns:
        validation_results_df['original_fuel_type'] = validation_results_df['original_fuel_type_pred']
    else:
        validation_results_df['original_fuel_type'] = 'UNKNOWN'

fuel_type_name_map = {'EL': 'BEV', 'HY': 'HEV'}
validation_results_df['fuel_type_group'] = validation_results_df['original_fuel_type'].map(fuel_type_name_map).fillna('OTHER')

validation_results_df['predicted_last_test'] = validation_results_df['last_test'].astype(int)
validation_results_df['actual_last_test'] = validation_results_df['actual_last_test'].astype(int)
validation_results_df['mileage_error'] = validation_results_df['mileage_per_year'] - validation_results_df['actual_mileage_per_year']
validation_results_df['absolute_mileage_error'] = validation_results_df['mileage_error'].abs()
validation_results_df['predicted_test_mileage_error'] = (
    validation_results_df['test_mileage']
    - validation_results_df['actual_test_mileage']
)
validation_results_df['predicted_test_mileage_rebased'] = (
    validation_results_df['actual_test_mileage']
    - validation_results_df['actual_mileage_per_year'] * validation_results_df['actual_time_between_tests']
    + validation_results_df['mileage_per_year'] * validation_results_df['actual_time_between_tests']
)
validation_results_df['rebased_test_mileage_error'] = (
    validation_results_df['predicted_test_mileage_rebased']
    - validation_results_df['actual_test_mileage']
)
validation_results_df['actual_positive'] = validation_results_df['actual_last_test'].astype(int)
validation_results_df['predicted_positive'] = (
    validation_results_df['scrap_probability'] >= validation_threshold
).astype(int)

print(f"Prediction rows created: {len(validation_predictions_df):,}")
print(f"Merged holdout rows: {len(validation_results_df):,}")

if len(validation_results_df) != len(validation_targets_df):
    missing_ids = set(validation_targets_df['vehicle_id']) - set(validation_results_df['vehicle_id'])
    print(f"Warning: {len(missing_ids):,} held-out vehicles were not scored")

# Regression metrics for mileage_per_year
regression_metrics = {
    'rmse': np.sqrt(mean_squared_error(
        validation_results_df['actual_mileage_per_year'],
        validation_results_df['mileage_per_year']
    )),
    'mae': mean_absolute_error(
        validation_results_df['actual_mileage_per_year'],
        validation_results_df['mileage_per_year']
    ),
    'median_ae': median_absolute_error(
        validation_results_df['actual_mileage_per_year'],
        validation_results_df['mileage_per_year']
    ),
    'r2': r2_score(
        validation_results_df['actual_mileage_per_year'],
        validation_results_df['mileage_per_year']
    )
}

# Classification metrics for last_test
classification_metrics = {
    'accuracy': accuracy_score(
        validation_results_df['actual_positive'],
        validation_results_df['predicted_positive']
    ),
    'precision': precision_score(
        validation_results_df['actual_positive'],
        validation_results_df['predicted_positive'],
        zero_division=0
    ),
    'recall': recall_score(
        validation_results_df['actual_positive'],
        validation_results_df['predicted_positive'],
        zero_division=0
    ),
    'f1': f1_score(
        validation_results_df['actual_positive'],
        validation_results_df['predicted_positive'],
        zero_division=0
    )
}

if validation_results_df['actual_positive'].nunique() > 1:
    classification_metrics['roc_auc'] = roc_auc_score(
        validation_results_df['actual_positive'],
        validation_results_df['scrap_probability']
    )
else:
    classification_metrics['roc_auc'] = np.nan

cm = confusion_matrix(
    validation_results_df['actual_positive'],
    validation_results_df['predicted_positive'],
    labels=[0, 1]
)

print("\nMileage prediction metrics (held-out final tests):")
for metric_name, metric_value in regression_metrics.items():
    print(f"\t{metric_name}: {metric_value:,.3f}")

print("\nScrappage classification metrics:")
print(f"\tThreshold: {validation_threshold:.2f}")
for metric_name, metric_value in classification_metrics.items():
    if pd.isna(metric_value):
        print(f"\t{metric_name}: NaN")
    else:
        print(f"\t{metric_name}: {metric_value:,.3f}")

print("\nConfusion matrix [actual rows 0/1, predicted columns 0/1]:")
print(cm)

print("\nAverage actual vs predicted values:")
print(f"\tActual mileage_per_year: {validation_results_df['actual_mileage_per_year'].mean():,.0f}")
print(f"\tPredicted mileage_per_year: {validation_results_df['mileage_per_year'].mean():,.0f}")
print(f"\tActual last_test rate: {validation_results_df['actual_last_test'].mean():.2%}")
print(f"\tPredicted last_test rate: {validation_results_df['predicted_positive'].mean():.2%}")
print(f"\tMean predicted scrap probability: {validation_results_df['scrap_probability'].mean():.2%}")

print("\nResults by original fuel code:")
subgroup_rows = []
for fuel in sorted(validation_results_df['original_fuel_type'].dropna().unique()):
    subgroup = validation_results_df[validation_results_df['original_fuel_type'] == fuel]
    subgroup_row = {
        'fuel': fuel,
        'n': len(subgroup),
        'rmse': np.sqrt(mean_squared_error(subgroup['actual_mileage_per_year'], subgroup['mileage_per_year'])),
        'mae': mean_absolute_error(subgroup['actual_mileage_per_year'], subgroup['mileage_per_year']),
        'actual_scrap_rate': subgroup['actual_last_test'].mean(),
        'predicted_scrap_rate': subgroup['predicted_positive'].mean(),
        'accuracy': accuracy_score(subgroup['actual_positive'], subgroup['predicted_positive'])
    }
    if subgroup['actual_positive'].nunique() > 1:
        subgroup_row['roc_auc'] = roc_auc_score(subgroup['actual_positive'], subgroup['scrap_probability'])
    else:
        subgroup_row['roc_auc'] = np.nan
    subgroup_rows.append(subgroup_row)

subgroup_metrics_df = pd.DataFrame(subgroup_rows)
print(subgroup_metrics_df.to_string(index=False))

print("\nResults by fuel type (BEV vs HEV):")
fuel_type_rows = []
for fuel_name in ['BEV', 'HEV']:
    subgroup = validation_results_df[validation_results_df['fuel_type_group'] == fuel_name]
    if len(subgroup) == 0:
        continue
    fuel_row = {
        'fuel_type': fuel_name,
        'n': len(subgroup),
        'rmse': np.sqrt(mean_squared_error(subgroup['actual_mileage_per_year'], subgroup['mileage_per_year'])),
        'mae': mean_absolute_error(subgroup['actual_mileage_per_year'], subgroup['mileage_per_year']),
        'median_ae': median_absolute_error(subgroup['actual_mileage_per_year'], subgroup['mileage_per_year']),
        'r2': r2_score(subgroup['actual_mileage_per_year'], subgroup['mileage_per_year']),
        'actual_scrap_rate': subgroup['actual_last_test'].mean(),
        'predicted_scrap_rate': subgroup['predicted_positive'].mean(),
        'accuracy': accuracy_score(subgroup['actual_positive'], subgroup['predicted_positive']),
        'precision': precision_score(subgroup['actual_positive'], subgroup['predicted_positive'], zero_division=0),
        'recall': recall_score(subgroup['actual_positive'], subgroup['predicted_positive'], zero_division=0),
        'f1': f1_score(subgroup['actual_positive'], subgroup['predicted_positive'], zero_division=0)
    }
    if subgroup['actual_positive'].nunique() > 1:
        fuel_row['roc_auc'] = roc_auc_score(subgroup['actual_positive'], subgroup['scrap_probability'])
    else:
        fuel_row['roc_auc'] = np.nan
    fuel_type_rows.append(fuel_row)

fuel_type_metrics_df = pd.DataFrame(fuel_type_rows)
print(fuel_type_metrics_df.to_string(index=False))

print("\nResults by fuel type (BEV vs HEV) and first_use_year_actual:")
fuel_type_rows = []
for fuel_name in ['BEV', 'HEV']:
    for year in range(2005, 2021):
        subgroup = validation_results_df[
            (validation_results_df['fuel_type_group'] == fuel_name)
            & (validation_results_df['first_use_year_actual'] == year)
        ]
        if len(subgroup) == 0:
            continue
        fuel_row = {
            'fuel_type': fuel_name,
            'first_use_year': year,
            'n': len(subgroup),
            'rmse': np.sqrt(mean_squared_error(subgroup['actual_mileage_per_year'], subgroup['mileage_per_year'])),
            'mae': mean_absolute_error(subgroup['actual_mileage_per_year'], subgroup['mileage_per_year']),
            'median_ae': median_absolute_error(subgroup['actual_mileage_per_year'], subgroup['mileage_per_year']),
            'r2': r2_score(subgroup['actual_mileage_per_year'], subgroup['mileage_per_year']),
            'actual_scrap_rate': subgroup['actual_last_test'].mean(),
            'predicted_scrap_rate': subgroup['predicted_positive'].mean(),
            'accuracy': accuracy_score(subgroup['actual_positive'], subgroup['predicted_positive']),
            'precision': precision_score(subgroup['actual_positive'], subgroup['predicted_positive'], zero_division=0),
            'recall': recall_score(subgroup['actual_positive'], subgroup['predicted_positive'], zero_division=0),
            'f1': f1_score(subgroup['actual_positive'], subgroup['predicted_positive'], zero_division=0)
        }
        if subgroup['actual_positive'].nunique() > 1:
            fuel_row['roc_auc'] = roc_auc_score(subgroup['actual_positive'], subgroup['scrap_probability'])
        else:
            fuel_row['roc_auc'] = np.nan
        fuel_type_rows.append(fuel_row)

fuel_type_metrics_df = pd.DataFrame(fuel_type_rows)
print(fuel_type_metrics_df.to_string(index=False))

print("\nThreshold sensitivity summary:")
threshold_rows = []
for threshold in [0.10, 0.22, 0.32, 0.4, 0.45, 0.50]:
    predicted_positive = (validation_results_df['scrap_probability'] >= threshold).astype(int)
    threshold_row = {
        'threshold': threshold,
        'accuracy': accuracy_score(validation_results_df['actual_positive'], predicted_positive),
        'precision': precision_score(validation_results_df['actual_positive'], predicted_positive, zero_division=0),
        'recall': recall_score(validation_results_df['actual_positive'], predicted_positive, zero_division=0),
        'f1': f1_score(validation_results_df['actual_positive'], predicted_positive, zero_division=0),
        'predicted_scrap_rate': predicted_positive.mean(),
    }
    threshold_rows.append(threshold_row)

threshold_metrics_df = pd.DataFrame(threshold_rows)
print(threshold_metrics_df.to_string(index=False))

print("\nLargest mileage errors:")
error_columns = [
    'vehicle_id', 'fuel_type_group', 'original_fuel_type', 'actual_test_year', 'observed_sequence_length',
    'actual_mileage_per_year', 'mileage_per_year', 'mileage_error', 'absolute_mileage_error',
    'actual_last_test', 'predicted_last_test', 'scrap_probability'
]
print(
    validation_results_df
    .sort_values('absolute_mileage_error', ascending=False)[error_columns]
    .head(20)
    .to_string(index=False)
)

validation_results_df.head()

In [ ]:
print("\nResults by fuel type (BEV vs HEV):")
fuel_type_rows = []
for fuel_name in ['BEV', 'HEV']:
    subgroup = validation_results_df[(validation_results_df['fuel_type_group'] == fuel_name) & (validation_results_df['first_use_year_actual'] != 2020)]
    
    
    if len(subgroup) == 0:
        continue
    fuel_row = {
        'fuel_type': fuel_name,
        'n': len(subgroup),
        'mean' : subgroup['mileage_per_year'].mean(),
        'rmse': np.sqrt(mean_squared_error(subgroup['actual_mileage_per_year'], subgroup['mileage_per_year'])),
        'mae': mean_absolute_error(subgroup['actual_mileage_per_year'], subgroup['mileage_per_year']),
        'median_ae': median_absolute_error(subgroup['actual_mileage_per_year'], subgroup['mileage_per_year']),
        'r2': r2_score(subgroup['actual_mileage_per_year'], subgroup['mileage_per_year']),
        'actual_scrap_rate': subgroup['actual_last_test'].mean(),
        'predicted_scrap_rate': subgroup['predicted_positive'].mean(),
        'accuracy': accuracy_score(subgroup['actual_positive'], subgroup['predicted_positive']),
        'precision': precision_score(subgroup['actual_positive'], subgroup['predicted_positive'], zero_division=0),
        'recall': recall_score(subgroup['actual_positive'], subgroup['predicted_positive'], zero_division=0),
        'f1': f1_score(subgroup['actual_positive'], subgroup['predicted_positive'], zero_division=0)
    }
    if subgroup['actual_positive'].nunique() > 1:
        fuel_row['roc_auc'] = roc_auc_score(subgroup['actual_positive'], subgroup['scrap_probability'])
    else:
        fuel_row['roc_auc'] = np.nan
    fuel_type_rows.append(fuel_row)

fuel_type_metrics_df = pd.DataFrame(fuel_type_rows)
print(fuel_type_metrics_df.to_string(index=False))

print("\nResults by fuel type (BEV vs HEV) and first_use_year_actual:")
fuel_type_rows = []
for fuel_name in ['BEV', 'HEV']:
    for year in range(2005, 2021):
        subgroup = validation_results_df[
            (validation_results_df['fuel_type_group'] == fuel_name)
            & (validation_results_df['first_use_year_actual'] == year)
        ]
        if len(subgroup) == 0:
            continue
        
        fuel_row = {
            'fuel_type': fuel_name,
            'first_use_year': year,
            'n': len(subgroup),
            'mean' : subgroup['mileage_per_year'].mean(),
            'rmse': np.sqrt(mean_squared_error(subgroup['actual_mileage_per_year'], subgroup['mileage_per_year'])),
            'mae': mean_absolute_error(subgroup['actual_mileage_per_year'], subgroup['mileage_per_year']),
            'median_ae': median_absolute_error(subgroup['actual_mileage_per_year'], subgroup['mileage_per_year']),
            'r2': r2_score(subgroup['actual_mileage_per_year'], subgroup['mileage_per_year']),
            'actual_scrap_rate': subgroup['actual_last_test'].mean(),
            'predicted_scrap_rate': subgroup['predicted_positive'].mean(),
            'accuracy': accuracy_score(subgroup['actual_positive'], subgroup['predicted_positive']),
            'precision': precision_score(subgroup['actual_positive'], subgroup['predicted_positive'], zero_division=0),
            'recall': recall_score(subgroup['actual_positive'], subgroup['predicted_positive'], zero_division=0),
            'f1': f1_score(subgroup['actual_positive'], subgroup['predicted_positive'], zero_division=0)
        }
        if subgroup['actual_positive'].nunique() > 1:
            fuel_row['roc_auc'] = roc_auc_score(subgroup['actual_positive'], subgroup['scrap_probability'])
        else:
            fuel_row['roc_auc'] = np.nan
        fuel_type_rows.append(fuel_row)

fuel_type_metrics_df = pd.DataFrame(fuel_type_rows)
print(fuel_type_metrics_df.to_string(index=False))

print("\nThreshold sensitivity summary:")
threshold_rows = []
for fuel_name in ['BEV', 'HEV']:
    for threshold in [0.10, 0.22, 0.32, 0.4, 0.45, 0.46, 0.47, 0.48, 0.49, 0.50]:
        subgroup = validation_results_df[(validation_results_df['first_use_year_actual'] != 2020) & (validation_results_df['fuel_type_group'] == fuel_name)]
        predicted_positive = (subgroup['scrap_probability'] >= threshold).astype(int)
        threshold_row = {
            'fuel': fuel_name,
            'threshold': threshold,
            'accuracy': accuracy_score(subgroup['actual_positive'], predicted_positive),
            'precision': precision_score(subgroup['actual_positive'], predicted_positive, zero_division=0),
            'recall': recall_score(subgroup['actual_positive'], predicted_positive, zero_division=0),
            'f1': f1_score(subgroup['actual_positive'], predicted_positive, zero_division=0),
            'roc_auc': roc_auc_score(subgroup['actual_positive'], subgroup['scrap_probability']),
            'actual_scrap_rate': subgroup['actual_last_test'].mean(),
            'predicted_scrap_rate': predicted_positive.mean(),
        }
        threshold_rows.append(threshold_row)

threshold_metrics_df = pd.DataFrame(threshold_rows)
print(threshold_metrics_df.to_string(index=False))